# Drift Case Study: The 2022 Policy-Shift Structural Break

The National Housing Accord (August 2022) was a major planning/governance policy event. Around the same time, post-COVID house construction costs peaked at ~15–20% YoY growth — far outside the pre-2022 training distribution. Any model trained on pre-2022 data faces a structural break at this point.

This notebook documents what that looks like in the monitoring layer:
1. Load the full ABS approvals time series
2. Fit the LSTM model through Q2 2022 (pre-break period)
3. Show predictions Q3 2022 onwards vs actuals — the miss becomes visible
4. Show feature drift flag timing (construction cost z-score breaching threshold)
5. Show residual drift flag timing (rolling MAE exceeding 1.5× baseline)

**Key finding:** The feature drift flag can trigger in the same quarter as the supply shock. Residual drift follows 1–2 quarters later, once actual approval data confirms the miss. That lag is the value of monitoring: early warning, not post-mortem.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler

from models.lstm import HousingLSTM, _make_sequences, fit as lstm_fit
from monitoring.drift import detect_feature_drift, _FEATURE_DRIFT_Z_THRESHOLD

FEATURES_PATH = Path('../data/processed/features.parquet')
TRAIN_END = '2022Q2'

## 1. Load full ABS time series

In [ ]:
df = pd.read_parquet(FEATURES_PATH)
df['quarter'] = pd.PeriodIndex(df['quarter'], freq='Q')
print(f"Data covers {df['quarter'].min()} to {df['quarter'].max()}")

# Aggregate to national level for visualisation
national = df.groupby('quarter').agg(
    approvals=('dwellings_approved', 'sum'),
    construction_cost_yoy=('construction_cost_yoy', 'first'),
    post_accord_2022=('post_accord_2022', 'first'),
).reset_index()
national['quarter_dt'] = national['quarter'].dt.to_timestamp()
national.tail(8)

## 2. Train LSTM through Q2 2022 (pre-break period)

In [ ]:
FEATURE_COLS = [
    'approvals_lag1',
    'population_growth_yoy',
    'construction_cost_yoy',
    'season_q1', 'season_q2', 'season_q3', 'season_q4',
    'post_accord_2022',
]
available = [c for c in FEATURE_COLS if c in df.columns]

train_mask = df['quarter'] <= TRAIN_END
val_mask = df['quarter'] > TRAIN_END

X_train_raw = df[train_mask][available].values.astype(np.float32)
y_train_raw = df[train_mask]['dwellings_approved'].values.astype(np.float32)
X_val_raw = df[val_mask][available].values.astype(np.float32)
y_val_raw = df[val_mask]['dwellings_approved'].values.astype(np.float32)

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()
X_train = x_scaler.fit_transform(X_train_raw)
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
X_val = x_scaler.transform(X_val_raw)
y_val = y_scaler.transform(y_val_raw.reshape(-1, 1)).ravel()

model = HousingLSTM(input_size=len(available), hidden_size=32, num_layers=1, output_steps=4)
history = lstm_fit(model, X_train, y_train, X_val, y_val,
                   seq_len=8, horizon=4, batch_size=16, max_epochs=30, patience=5)

print(f"Training complete. Best val MAE: {history['best_val_mae']:.4f}")

## 3. Show predictions Q3 2022 onwards vs actuals

In [ ]:
model.eval()
device = next(model.parameters()).device

X_vl_seq, y_vl_seq = _make_sequences(X_val, y_val, seq_len=8, horizon=4)
with torch.no_grad():
    preds_scaled = model(X_vl_seq.to(device)).cpu().numpy()

preds = y_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).reshape(preds_scaled.shape)
actuals = y_scaler.inverse_transform(y_vl_seq.numpy().reshape(-1, 1)).reshape(y_vl_seq.shape)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(actuals.mean(axis=1), label='Actual (mean across LGAs)', color='steelblue', marker='o')
ax.plot(preds.mean(axis=1), label='Predicted', color='darkorange', linestyle='--', marker='x')
ax.set_title('Model Predictions vs Actuals: Post-Accord Period (Q3 2022 onwards)')
ax.set_xlabel('Sequence index (quarters after Q3 2022)')
ax.set_ylabel('Dwellings approved (scaled mean)')
ax.legend()
plt.tight_layout()
plt.show()

overall_mae = np.mean(np.abs(actuals - preds))
print(f"Post-Accord MAE: {overall_mae:.4f} (compare to training MAE)")

## 4. Feature drift flag: construction cost z-score over time

In [ ]:
train_costs = df[df['post_accord_2022'] == 0]['construction_cost_yoy'].dropna()
mean_cost = train_costs.mean()
std_cost = train_costs.std()

macro_sorted = national.sort_values('quarter')
macro_sorted['z_score'] = (macro_sorted['construction_cost_yoy'] - mean_cost) / (std_cost if std_cost > 0 else 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(macro_sorted['quarter_dt'], macro_sorted['z_score'],
        color='steelblue', label='Construction cost z-score')
ax.axhline(_FEATURE_DRIFT_Z_THRESHOLD, color='red', linestyle='--',
           label=f'Drift threshold (z={_FEATURE_DRIFT_Z_THRESHOLD})')
ax.axhline(-_FEATURE_DRIFT_Z_THRESHOLD, color='red', linestyle='--')
ax.axhline(0, color='green', linestyle=':', alpha=0.5, label='Training mean')

breach_mask = macro_sorted['z_score'].abs() > _FEATURE_DRIFT_Z_THRESHOLD
first_breach = macro_sorted[breach_mask]['quarter_dt'].min()
if pd.notna(first_breach):
    label = f'Feature drift triggered: {first_breach.year} Q{first_breach.quarter}'
    ax.axvline(first_breach, color='darkorange', linewidth=2, linestyle=':', label=label)

ax.set_title('Feature Drift: Construction Cost Z-Score vs Pre-Accord Training Distribution')
ax.set_ylabel('Z-score')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Residual drift: rolling MAE over time

In [ ]:
# Compute per-sequence MAE to simulate rolling residual monitoring
seq_mae = np.abs(actuals - preds).mean(axis=1)
training_mae = history['best_val_mae'] * (y_train_raw.max() - y_train_raw.min())  # approximate unscaled
drift_threshold = 1.5  # 1.5x training MAE

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(seq_mae)), seq_mae, color='steelblue', label='Per-quarter MAE')

# Rolling mean
rolling = pd.Series(seq_mae).rolling(4).mean()
ax.plot(range(len(rolling)), rolling, color='steelblue', linestyle='--', linewidth=2, label='Rolling 4-quarter MAE')

baseline = seq_mae[:4].mean() if len(seq_mae) >= 4 else seq_mae.mean()
ax.axhline(baseline, color='green', linestyle=':', label=f'Baseline MAE (first 4 seqs): {baseline:.4f}')
ax.axhline(baseline * drift_threshold, color='red', linestyle='--', label=f'Alert threshold (1.5x): {baseline * drift_threshold:.4f}')

ax.set_title('Residual Drift: Rolling MAE in Post-Rate-Hike Period')
ax.set_xlabel('Quarter index (after Q3 2022)')
ax.set_ylabel('MAE (scaled)')
ax.legend()
plt.tight_layout()
plt.show()

## Business Impact

The 2022 policy-shift is a textbook concept drift case with two compounding factors:

- **Feature drift** (input distribution shift) triggers when house construction cost growth moves outside the training distribution. Post-COVID supply chain stress pushed house construction PPI to ~15–20% YoY growth — well above the pre-2022 training baseline of ~1–5%. The feature drift flag can fire in the same quarter, without requiring ground truth.
- **Residual drift** (degraded prediction quality) follows 1–2 quarters later, once actual approval data confirms the forecast miss.

The production value is the **lag between signals**: feature drift flags the problem before you have ground truth. In a housing market context, a false sense of stability lasts only until the first actuals arrive — roughly 1 quarter behind. A monitoring layer would have alerted a retraining pipeline in the quarter the cost shock occurred, rather than discovering the problem months later when the miss became visible in reports.

For an infrastructure planning team tracking dwelling supply against the National Housing Accord target (1.2 million homes in 5 years), a model that silently degrades over a 6-month period is operationally damaging. Early warning enables:
1. Retraining on updated data before forecast quality visibly drops
2. Flagging model uncertainty to downstream users while retraining is underway
3. Informing decision-makers that the structural environment has changed, not just that forecasts are temporarily unreliable